In [2]:
import pandas as pd
from Bio import SeqIO

# Function to read a FASTA file
def read_fasta(fasta_file):
    sequences = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences[record.id] = str(record.seq)
    return sequences

# Function to load the MHC epitope data from CSV
def load_epitope_data(epitope_file):
    # Read the CSV file into a DataFrame
    df = pd.read_csv(epitope_file)
    return df

# Function to map vaccine peptides to MHC epitopes
def map_peptides_to_epitopes(vaccine_sequences, mhc_data, dataset_type):
    mapped_data = []
    for vaccine_id, vaccine_seq in vaccine_sequences.items():
        for epitope_id, epitope_seq in mhc_data['Peptide'].items():
            if epitope_seq in vaccine_seq:  # Check if the epitope is present in the vaccine sequence
                # Get additional information from MHC dataset
                epitope_info = mhc_data[mhc_data['Peptide'] == epitope_seq]
                mapped_data.append({
                    'Vaccine': vaccine_id,
                    'Peptide': epitope_seq,
                    'Dataset': dataset_type,
                    **epitope_info.to_dict(orient='records')[0]  # Get row data as dictionary
                })
    return mapped_data

# Function to extract linkers
def extract_linkers(vaccine_sequences):
    linkers = []
    for vaccine_id, vaccine_seq in vaccine_sequences.items():
        linker_positions = [(i, seq) for i, seq in enumerate(vaccine_seq) if seq in ['EAAAK', 'GPGPG']]
        for position, linker in linker_positions:
            linkers.append({
                'Vaccine': vaccine_id,
                'Linker': linker,
                'Position': position
            })
    return linkers

# Paths to your files (adjust to the correct paths on your system)
vaccine_fasta_path = 'noadjuvants.fasta'  # Path to vaccine FASTA file
mhc_i_file = 'Unique_MHC_I_Epitopes.csv'  # Path to MHC I epitopes CSV file
mhc_ii_file = 'Unique_MHC_II_Epitopes.csv'  # Path to MHC II epitopes CSV file

# Load vaccine sequences and MHC epitope data
vaccine_sequences = read_fasta(vaccine_fasta_path)
mhc_i_data = load_epitope_data(mhc_i_file)
mhc_ii_data = load_epitope_data(mhc_ii_file)

# Map vaccine peptides to MHC I and MHC II epitopes
mapped_mhc_i = map_peptides_to_epitopes(vaccine_sequences, mhc_i_data, "MHC I")
mapped_mhc_ii = map_peptides_to_epitopes(vaccine_sequences, mhc_ii_data, "MHC II")

# Combine MHC I and MHC II results
mapped_data = mapped_mhc_i + mapped_mhc_ii

# Convert mapped data to a DataFrame
mapped_df = pd.DataFrame(mapped_data)

# Extract linker data
linker_data = extract_linkers(vaccine_sequences)
linker_df = pd.DataFrame(linker_data)

# Save results to CSV files
mapped_df.to_csv('mapped_vaccine_to_epitopes.csv', index=False)
linker_df.to_csv('vaccine_linkers.csv', index=False)

# Display results
print("Mapped Data:")
print(mapped_df.head())
print("\nLinker Data:")
print(linker_df.head())


KeyError: 'Peptide'

In [3]:
# Function to map vaccine peptides to MHC epitopes using the correct column names
def map_peptides_to_epitopes(vaccine_sequences, mhc_data, dataset_type, peptide_column='peptide'):
    mapped_data = []
    for vaccine_id, vaccine_seq in vaccine_sequences.items():
        for epitope_id, epitope_seq in mhc_data[peptide_column].items():
            if epitope_seq in vaccine_seq:  # Check if the epitope is present in the vaccine sequence
                # Get additional information from MHC dataset
                epitope_info = mhc_data[mhc_data[peptide_column] == epitope_seq]
                mapped_data.append({
                    'Vaccine': vaccine_id,
                    'Peptide': epitope_seq,
                    'Dataset': dataset_type,
                    **epitope_info.to_dict(orient='records')[0]  # Get row data as dictionary
                })
    return mapped_data

# Map vaccine peptides to MHC I and MHC II epitopes with the correct column names
mapped_mhc_i = map_peptides_to_epitopes(vaccine_sequences, mhc_i_data, "MHC I", peptide_column='peptide')
mapped_mhc_ii = map_peptides_to_epitopes(vaccine_sequences, mhc_ii_data, "MHC II", peptide_column='peptide')

# Combine MHC I and MHC II results
mapped_data = mapped_mhc_i + mapped_mhc_ii

# Convert mapped data to a DataFrame
mapped_df = pd.DataFrame(mapped_data)

# Extract linker data
linker_data = extract_linkers(vaccine_sequences)
linker_df = pd.DataFrame(linker_data)

# Save results to CSV files
mapped_df.to_csv('mapped_vaccine_to_epitopes.csv', index=False)
linker_df.to_csv('vaccine_linkers.csv', index=False)

# Display results
print("Mapped Data:")
print(mapped_df.head())
print("\nLinker Data:")
print(linker_df.head())


Mapped Data:
    Vaccine    Peptide Dataset       allele  seq_num  start  end  length  \
0  Vaccine1  LPQGQLTAY   MHC I  HLA-B*35:01        2     56   64       9   
1  Vaccine1  TMFEVSVAF   MHC I  HLA-B*15:01        6    290  298       9   
2  Vaccine1  FLDKGTYTL   MHC I  HLA-A*02:01        2    276  284       9   
3  Vaccine1  RVLSVVVLL   MHC I  HLA-A*32:01        2      5   13       9   
4  Vaccine1  RPQKRPSCI   MHC I  HLA-B*07:02        3     72   80       9   

     peptide       core      icore     score  rank  protein_source  \
0  LPQGQLTAY  LPQGQLTAY  LPQGQLTAY  0.990098  0.01             NaN   
1  TMFEVSVAF  TMFEVSVAF  TMFEVSVAF  0.948171  0.01             NaN   
2  FLDKGTYTL  FLDKGTYTL  FLDKGTYTL  0.996447  0.01             4.0   
3  RVLSVVVLL  RVLSVVVLL  RVLSVVVLL  0.811834  0.02             4.0   
4  RPQKRPSCI  RPQKRPSCI  RPQKRPSCI  0.907598  0.04             NaN   

  core_peptide  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  

Linker

In [4]:
from collections import Counter

# Function to map vaccine peptides to MHC epitopes and track non-matching peptides
def map_peptides_to_epitopes_with_non_matching(vaccine_sequences, mhc_i_data, mhc_ii_data):
    mapped_data = []
    
    # Iterate through each vaccine sequence
    for vaccine_id, vaccine_seq in vaccine_sequences.items():
        found = False
        # Check MHC I epitopes
        for epitope_seq in mhc_i_data['peptide']:
            if epitope_seq in vaccine_seq:  # Check if the epitope is present in the vaccine sequence
                epitope_info = mhc_i_data[mhc_i_data['peptide'] == epitope_seq]
                mapped_data.append({
                    'Vaccine': vaccine_id,
                    'Peptide': epitope_seq,
                    'Dataset': "MHC I",
                    **epitope_info.to_dict(orient='records')[0]
                })
                found = True
                break
        
        # Check MHC II epitopes if not found in MHC I
        if not found:
            for epitope_seq in mhc_ii_data['peptide']:
                if epitope_seq in vaccine_seq:  # Check if the epitope is present in the vaccine sequence
                    epitope_info = mhc_ii_data[mhc_ii_data['peptide'] == epitope_seq]
                    mapped_data.append({
                        'Vaccine': vaccine_id,
                        'Peptide': epitope_seq,
                        'Dataset': "MHC II",
                        **epitope_info.to_dict(orient='records')[0]
                    })
                    found = True
                    break
        
        # If no match was found, label as 'non'
        if not found:
            mapped_data.append({
                'Vaccine': vaccine_id,
                'Peptide': vaccine_seq,
                'Dataset': 'non'
            })
    
    return mapped_data

# Function to extract and count linkers and their positions
def extract_and_count_linkers(vaccine_sequences):
    linkers = []
    linker_counts = Counter()  # To track the frequency of each linker
    for vaccine_id, vaccine_seq in vaccine_sequences.items():
        # Search for the linkers in the vaccine sequence
        for linker in ['EAAAK', 'GPGPG']:
            positions = [i for i in range(len(vaccine_seq)) if vaccine_seq.startswith(linker, i)]
            for pos in positions:
                linkers.append({
                    'Vaccine': vaccine_id,
                    'Linker': linker,
                    'Position': pos
                })
                linker_counts[linker] += 1
    
    return linkers, linker_counts

# Load vaccine sequences and epitope data
vaccine_sequences = read_fasta(vaccine_fasta_path)
mapped_data = map_peptides_to_epitopes_with_non_matching(vaccine_sequences, mhc_i_data, mhc_ii_data)

# Convert mapped data to DataFrame
mapped_df = pd.DataFrame(mapped_data)

# Extract and count linkers
linker_data, linker_counts = extract_and_count_linkers(vaccine_sequences)
linker_df = pd.DataFrame(linker_data)

# Save results to CSV files
mapped_df.to_csv('mapped_vaccine_to_epitopes_with_non_matching.csv', index=False)
linker_df.to_csv('vaccine_linkers_with_counts.csv', index=False)

# Display results
print("Mapped Data (with non-matching peptides):")
print(mapped_df.head())
print("\nLinker Data (with positions and counts):")
print(linker_df.head())
print("\nLinker Frequencies:")
print(linker_counts)


Mapped Data (with non-matching peptides):
     Vaccine    Peptide Dataset       allele  seq_num  start  end  length  \
0   Vaccine1  LPQGQLTAY   MHC I  HLA-B*35:01        2     56   64       9   
1  Vaccine10  LPQGQLTAY   MHC I  HLA-B*35:01        2     56   64       9   
2  Vaccine11  VPKPNVEVW   MHC I  HLA-B*53:01        8     45   53       9   
3  Vaccine12  GTYKRVTEK   MHC I  HLA-A*30:01        6    180  188       9   
4  Vaccine13  GTYKRVTEK   MHC I  HLA-A*30:01        6    180  188       9   

     peptide       core      icore     score  rank  protein_source  
0  LPQGQLTAY  LPQGQLTAY  LPQGQLTAY  0.990098  0.01             NaN  
1  LPQGQLTAY  LPQGQLTAY  LPQGQLTAY  0.990098  0.01             NaN  
2  VPKPNVEVW  VPKPNVEVW  VPKPNVEVW  0.972539  0.01             NaN  
3  GTYKRVTEK  GTYKRVTEK  GTYKRVTEK  0.895470  0.01             NaN  
4  GTYKRVTEK  GTYKRVTEK  GTYKRVTEK  0.895470  0.01             NaN  

Linker Data (with positions and counts):
    Vaccine Linker  Position
0  Vaccine